# ch09 练习：分箱与离散化

与讲解版逐 Cell 对应，`assert` 验收保留。卡住回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 数据探索 | `value_counts(bins=)` 快速体检 |
| 数据准备及处理 | `cut` / `qcut` 离散化 + 分层统计 |
| 数据清洗 | 按档位圈异常行 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

dev = pd.read_csv(DATA / "device_defects.csv")
load = pd.read_csv(DATA / "load_curve.csv")

VAL = "负荷值"
KEY = "台区编号"
CAT = "设备类型"
s = dev[VAL]

print("pandas", pd.__version__)
print("dev", dev.shape, "| 负荷值 NaN", int(s.isna().sum()),
      "| min", s.min(), "| max", s.max())

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


# 演示 right 语义用的合成序列：正好踩在边界 100 / 300 / 500 上
DEMO = pd.Series([100, 200, 300, 400, 500], name="演示")
EDGES = [-10, 100, 300, 600, 10000]
LABELS = ["低", "中", "高", "极端"]

print("DEMO", DEMO.tolist(), "| 边界", EDGES)

## 题 1：等宽分箱体检（对应讲解 §3.1）

In [ ]:
# TODO(1)：对负荷值 s 做 5 箱等宽分箱，结果存入 b5
# 提示：pd.cut(s, 5)
b5 = ____

for iv in b5.cat.categories:
    print("  ", iv, "| 计数", int((b5 == iv).sum()))
print("NaN:", int(b5.isna().sum()), "| ordered:", b5.cat.ordered)

assert len(b5.cat.categories) == 5
assert b5.value_counts().sort_index().tolist() == [242, 0, 3, 0, 4]
assert int(b5.isna().sum()) == 21

**为什么这样做**：两箱计数为 0 不是 bug——是 9999 这个离群值把等宽箱撑爆的直接证据，也是下一题换自定义边界的理由。

## 题 2：right 语义与越界（对应讲解 §3.2）

In [ ]:
# TODO(2)：对 DEMO 以边界 [100, 300, 500]、labels=["a","b"] 分箱（默认 right），存入 r
r = ____

# TODO(3)：同样的边界但 right=False，存入 rl
rl = ____

print("right=True :", r.tolist())
print("right=False:", rl.tolist())
out = pd.cut(pd.Series([50, 20000]), bins=[100, 300, 500])
print("箱外值:", out.tolist())

assert r.tolist() == [np.nan, "a", "a", "b", "b"]
assert rl.tolist() == ["a", "a", "b", "b", np.nan]
assert bool(out.isna().all())

## 题 3：业务边界四档 + 整数编码（对应讲解 §3.3）

In [ ]:
# TODO(4)：以 EDGES 边界、LABELS 标签对 s 分箱，存入 bin_col；计数打印
# 提示：pd.cut(s, bins=EDGES, labels=LABELS)
bin_col = ____

# TODO(5)：再来一版 labels=False 的整数编码，存入 bin_code
bin_code = ____

print("计数:", bin_col.value_counts().to_dict())
print("编码分布:", bin_code.value_counts().sort_index().to_dict())

assert bin_col.value_counts().to_dict() == {"低": 141, "中": 56, "高": 45, "极端": 7}
assert int(bin_col.isna().sum()) == 21
assert sorted(bin_code.dropna().unique().tolist()) == [0, 1, 2, 3]

## 题 4：qcut 分位分箱（对应讲解 §3.4）

In [ ]:
# TODO(6)：对 s 做 4 分位分箱存入 q4；再做 10 分位 duplicates="drop" 存入 q10
# 提示：pd.qcut(s, 4) ；pd.qcut(s, 10, duplicates="drop")
q4 = ____
q10 = ____

print("qcut(4) 边界:", [round(iv.right, 2) for iv in q4.cat.categories])
print("qcut(4) 计数:", q4.value_counts().sort_index().tolist())
print("qcut(10) 箱数:", len(q10.cat.categories))
status, msg = probe(pd.qcut, pd.Series([1, 1, 1, 1, 1, 1, 2, 3]), 4)
print("重复分位 →", status)

assert q4.value_counts().sort_index().tolist() == [63, 62, 62, 62]
assert [round(iv.right, 2) for iv in q4.cat.categories] == [48.31, 87.34, 192.34, 9999.0]
assert int(q4.isna().sum()) == 21
assert len(q10.cat.categories) == 10
assert status == "err"

## 题 5：分箱计数体检（对应讲解 §3.5）

In [ ]:
# TODO(7)：用 value_counts 的分箱参数直接对 s 做 5 箱计数，存入 vc
# 提示：s.value_counts(bins=5)
vc = ____

print("索引类型:", type(vc.index).__name__)
print("计数:", vc.sort_index().tolist())

assert type(vc.index).__name__ == "IntervalIndex"
assert vc.sort_index().tolist() == [242, 0, 3, 0, 4]

## 题 6：分档分层统计 + 档位过滤（对应讲解 §3.6-3.7）

In [ ]:
# TODO(8)：把题 3 的 bin_col 存为 dev["负荷档"]，再按负荷档分组统计
#            负荷值的 count/mean，存入 g（observed=False）
dev["负荷档"] = ____
g = ____

# TODO(9)：高负荷行数（档 > "中"）存入 n_hi；高或缺失行数存入 n_hi_na
# 提示：(dev["负荷档"] > "中") ；缺失要显式补 | dev["负荷档"].isna()
n_hi = ____
n_hi_na = ____

print(g.round(2))
print("高负荷:", n_hi, "行 | 高或缺失:", n_hi_na, "行")
assert g["count"].tolist() == [141, 56, 45, 7]
assert [round(v, 2) for v in g["mean"]] == [52.09, 157.83, 519.16, 7856.57]
assert n_hi == 52 and n_hi_na == 73

## 题 7（综合）：日均负荷四分位档（对应讲解 §3.8）

In [ ]:
# TODO(10)：load_curve 按天聚合负荷均值存入 daily（先把时间戳 to_datetime）；
#            再 qcut 成 Q1~Q4 四档存入 q_label
# 提示：lc["时间戳"].dt.date ；pd.qcut(daily, 4, labels=["Q1","Q2","Q3","Q4"])
lc = ____
lc["时间戳"] = ____
daily = ____
q_label = ____

print("日均负荷 n:", len(daily), "| min", round(daily.min(), 2),
      "| max", round(daily.max(), 2))
print("四分位档计数:", q_label.value_counts().to_dict())

assert len(daily) == 90
assert round(daily.min(), 2) == 424.68 and round(daily.max(), 2) == 778.21
assert q_label.value_counts().to_dict() == {"Q1": 23, "Q2": 22, "Q3": 22, "Q4": 23}

---

## 综合自查

1. 题 1 两箱为 0 说明什么？数字证据是什么？
2. 题 2 的 100 和 500 分别在哪种语义下出界？
3. 题 4 为什么 qcut 对偏态数据更稳？
4. 题 6 里 NaN 行是怎么被比较过滤天然排除的？想保留它们怎么写？

全部答得上来，进入 ch15（多级索引）。